# Задание №8. Построение нейронной сети для выявления спама (классификация текстовых сообщений с помощью LSTM)

Данное задание представляет собой пошаговое руководство по созданию модели для классификации текстовых сообщений на спам и не‑спам (ham) с использованием рекуррентной нейронной сети LSTM. Вы познакомитесь с основными понятиями обработки естественного языка (NLP), научитесь подготавливать размеченный корпус сообщений, строить и обучать модель, а также использовать её для предсказания класса новых сообщений. В задании обязательным требованием является фиксация времени обучения модели – вы должны добавить в код соответствующие замеры и вывести результат. Задание адаптировано для выполнения в **Google Colab** (Jupyter Notebook). Все части работы (подготовка данных, обучение, сохранение, загрузка из репозитория и инференс) выполняются в одном ноутбуке.

---

## 1. Теоретическое введение: ключевые понятия классификации спама

### 1.1. Задача выявления спама
Спам – это нежелательные сообщения, часто рассылаемые в больших объёмах. Автоматическое определение спама позволяет фильтровать входящие сообщения, повышая безопасность и удобство пользователей. Задача представляет собой **бинарную классификацию**: сообщение относится к одному из двух классов – спам (`spam`) или не‑спам (`ham`).

### 1.2. Корпус данных
Для обучения необходим размеченный корпус, содержащий примеры сообщений с метками. Наиболее популярный датасет – **SMS Spam Collection** (UCI Machine Learning Repository), который содержит около 5 500 SMS с пометками «spam» и «ham».

### 1.3. Токенизация (Tokenization)
Процесс разбиения текста на минимальные единицы – **токены** (слова, знаки препинания). Для токенизации используем `Tokenizer` из Keras, который преобразует текст в последовательность целочисленных индексов.

### 1.4. Эмбеддинги (Embeddings)
Слой **Embedding** преобразует индексы токенов в плотные векторы фиксированной размерности. Эти векторы обучаются вместе с моделью и позволяют улавливать семантические и синтаксические сходства между словами.

### 1.5. Рекуррентные нейронные сети (RNN, LSTM)
RNN – класс сетей, способных обрабатывать последовательности произвольной длины благодаря внутренней памяти. **LSTM (Long Short‑Term Memory)** – разновидность RNN, которая эффективно обучается на длинных последовательностях, избегая проблемы затухания градиента. В задаче выявления спама LSTM учитывает контекст и последовательность слов, что позволяет выявлять характерные спамерские фразы.

### 1.6. Метрики качества
Для бинарной классификации используют:
- **Accuracy** – доля правильных ответов.
- **Precision (точность)** – доля истинно‑положительных среди всех предсказанных положительных.
- **Recall (полнота)** – доля найденных положительных примеров.
- **F1‑score** – гармоническое среднее precision и recall.
- **AUC‑ROC** – площадь под ROC‑кривой.

### 1.7. Фиксация времени обучения
Для оценки производительности и воспроизводимости эксперимента важно фиксировать время, затраченное на обучение модели. В коде необходимо использовать модуль `time` для замера длительности обучения и вывода результата.

---

## 2. Постановка задачи

Вам необходимо создать собственный репозиторий на GitHub, загрузить в него размеченный корпус сообщений (файл `spam.csv` или `sms.tsv`), затем построить и обучить модель LSTM для бинарной классификации сообщений на спам и не‑спам. **Обязательное требование:** в коде обучения модели нужно замерить время, затраченное на обучение, и вывести его в консоль (например, с помощью `time.time()`). После обучения вы сохраните модель, токенизатор и (при необходимости) кодировщик меток, загрузите их в тот же репозиторий. В финальной части ноутбука вы продемонстрируете, как загрузить эти файлы из репозитория и использовать их для предсказания класса новых сообщений без повторного обучения. Весь код и отчёт должны быть оформлены в одном Jupyter Notebook.

Формат файла с данными: текстовый файл с разделителем `,` или `\t`, содержащий колонки `label` (spam/ham) и `message` (текст). Пример строки из популярного датасета:
```
ham,Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005...
```

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы (в виде ячеек Markdown в ноутбуке):

1. **Титульный лист** (название работы, ФИО, группа, ссылка на репозиторий)
2. **Введение** (цель работы, краткое описание задачи выявления спама)
3. **Теоретическая часть** (объяснение ключевых понятий: спам, классификация, токенизация, эмбеддинги, LSTM, метрики)
4. **Описание данных** (источник данных, статистика: количество примеров, распределение по классам, средняя длина сообщения; ссылка на файл в репозитории)
5. **Подготовка данных** (загрузка, предобработка, токенизация, паддинг, разделение на train/test)
6. **Построение модели** (архитектура LSTM, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, использование early stopping, **фиксация времени обучения**, графики потерь и точности)
8. **Оценка качества** (расчёт accuracy, precision, recall, F1, матрица ошибок)
9. **Функция предсказания** (описание функции `predict_spam`, демонстрация примеров)
10. **Сохранение модели и вспомогательных объектов** (код сохранения модели, токенизатора; загрузка файлов в репозиторий)
11. **Загрузка модели из репозитория и быстрый инференс** (код загрузки через raw‑ссылку, повторное использование для предсказания)
12. **Выводы** (что получилось, какие были трудности, возможные улучшения)
13. **Список использованных источников**
14. **Приложение** (полный код с комментариями)

---

## 4. Источники данных для выявления спама

### 4.1. Открытые датасеты

| Источник | Описание | Ссылка |
|----------|----------|--------|
| **SMS Spam Collection (UCI)** | Классический датасет из 5 574 SMS, 13% спама | https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection |
| **Kaggle: SMS Spam Collection** | Та же коллекция в удобном формате CSV | https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset |
| **Enron Spam Dataset** | Электронные письма (спам/не‑спам), 33 000+ сообщений | https://www.kaggle.com/datasets/wcukierski/enron-spam |
| **Lingspam** | Корпус писем, хорошо сбалансированный | Поиск в открытых источниках |

### 4.2. Синтетический датасет (для тренировки)

Если у вас нет возможности скачать реальные данные, вы можете сгенерировать синтетический датасет с помощью простого шаблонизатора. Однако для реального выполнения задания рекомендуется использовать реальный датасет.

```python
import pandas as pd
import random

# Создаём простые шаблоны
ham_templates = [
    "Привет, как дела?", "Встреча завтра в 10.", "Спасибо за информацию!",
    "Позвони мне позже.", "Скинь фотографии с отпуска."
]
spam_templates = [
    "Вы выиграли миллион! Перейдите по ссылке.", "Срочно! Ваш аккаунт заблокирован.",
    "Получите скидку 50% сегодня.", "Заработай лёгкие деньги!",
    "Пришлите смс на номер 1234 для активации."
]

data = []
for _ in range(500):
    data.append([random.choice(ham_templates), "ham"])
for _ in range(500):
    data.append([random.choice(spam_templates), "spam"])

random.shuffle(data)
df = pd.DataFrame(data, columns=["message", "label"])
df.to_csv("spam_data.csv", index=False)
```

### 4.3. Требования к объёму корпуса
- **Минимальный объём**: не менее 1 000 сообщений, сбалансированных или с естественным распределением.
- **Максимальный объём**: в Google Colab ограничение по оперативной памяти; до 100 000 сообщений – комфортно.

### 4.4. Создание репозитория и загрузка данных
1. Зарегистрируйтесь на [GitHub](https://github.com).
2. Создайте новый публичный репозиторий с названием, например, `spam-detector-lstm`.
3. Загрузите в репозиторий файл с данными (например, `spam_data.csv`).
4. Получите **raw‑ссылку** на файл (открыть файл → Raw → скопировать URL). Пример:  
   `https://raw.githubusercontent.com/ваш_логин/spam-detector-lstm/main/spam_data.csv`

---

## 5. Пошаговое выполнение задания в Google Colab

### 5.1. Подготовка окружения

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import pickle
import requests
import io
```

### 5.2. Загрузка данных из репозитория

```python
# Вставьте вашу ссылку
url_data = "https://raw.githubusercontent.com/ваш_логин/spam-detector-lstm/main/spam_data.csv"

response = requests.get(url_data)
df = pd.read_csv(io.StringIO(response.text))

print(f"Загружено примеров: {len(df)}")
print(df['label'].value_counts())
df.head()
```

### 5.3. Предобработка текста и подготовка данных

Выполним базовую очистку текста (приведение к нижнему регистру, удаление лишних символов).

```python
def clean_text(text):
    text = str(text).lower()
    # Можно добавить более сложную очистку: удаление пунктуации, стоп-слов и т.д.
    return text

df['clean_message'] = df['message'].apply(clean_text)

# Параметры токенизации
VOCAB_SIZE = 10000  # максимальный размер словаря
MAX_LENGTH = 100    # максимальная длина текста (в словах)

# Токенизация
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_message'])
sequences = tokenizer.texts_to_sequences(df['clean_message'])
X = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Кодирование меток (spam → 1, ham → 0)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])
num_classes = 1  # бинарная классификация

print(f"Метки: {label_encoder.classes_}")
print(f"Форма X: {X.shape}")
```

### 5.4. Разделение на обучающую и тестовую выборки

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающих примеров: {len(X_train)}")
print(f"Тестовых примеров: {len(X_test)}")
```

### 5.5. Построение модели LSTM

```python
EMBEDDING_DIM = 100
LSTM_UNITS = 64

model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    LSTM(LSTM_UNITS, dropout=0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()
```

### 5.6. Обучение модели с фиксацией времени

**Обязательное требование:** добавьте код для измерения времени обучения и выведите его.

```python
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print("Начало обучения...")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time
print(f"\nОбучение завершено за {training_time:.2f} секунд ({(training_time/60):.2f} минут)")
```

### 5.7. Визуализация процесса обучения

```python
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери (Loss)')

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность (Accuracy)')
plt.show()
```

### 5.8. Оценка качества модели

```python
# Предсказания на тестовой выборке
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Метрики
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-score: {f1:.4f}")

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Матрица ошибок')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.show()
```

### 5.9. Функция предсказания

```python
def predict_spam(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = model.predict(padded, verbose=0)[0][0]
    pred_class = (prob > 0.5).astype(int)
    return label_encoder.inverse_transform([pred_class])[0], prob

# Примеры
test_messages = [
    "Поздравляем! Вы выиграли айфон. Перейдите по ссылке: http://bit.ly/...",
    "Встреча переносится на среду, извините за неудобства.",
    "Ваш аккаунт будет заблокирован, подтвердите данные немедленно."
]

for msg in test_messages:
    label, prob = predict_spam(msg)
    print(f"Сообщение: {msg}")
    print(f"Класс: {label}, вероятность спама: {prob:.4f}\n")
```

### 5.10. Сохранение модели и вспомогательных объектов

```python
# Сохраняем модель
model.save('spam_model.h5')
print("Модель сохранена в spam_model.h5")

# Сохраняем токенизатор
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("Токенизатор сохранён в tokenizer.pickle")

# Сохраняем label encoder
with open('label_encoder.pickle', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Label encoder сохранён в label_encoder.pickle")
```

### 5.11. Загрузка файлов в репозиторий

1. Скачайте файлы `spam_model.h5`, `tokenizer.pickle`, `label_encoder.pickle` на свой компьютер.
2. В своём репозитории на GitHub загрузите эти файлы.
3. Получите raw‑ссылки на каждый файл (формат `https://github.com/ваш_логин/spam-detector-lstm/raw/main/...`).

### 5.12. Загрузка модели из репозитория и быстрый инференс

```python
# Вставьте ваши ссылки
url_model = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/spam_model.h5"
url_tokenizer = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/tokenizer.pickle"
url_encoder = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/label_encoder.pickle"

# Скачиваем
!wget -O spam_model.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}
!wget -O label_encoder.pickle {url_encoder}

# Загружаем
loaded_model = load_model('spam_model.h5')
with open('tokenizer.pickle', 'rb') as f:
    loaded_tokenizer = pickle.load(f)
with open('label_encoder.pickle', 'rb') as f:
    loaded_encoder = pickle.load(f)

# Функция предсказания с загруженной моделью
def predict_loaded(text):
    cleaned = clean_text(text)
    seq = loaded_tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = loaded_model.predict(padded, verbose=0)[0][0]
    pred_class = (prob > 0.5).astype(int)
    return loaded_encoder.inverse_transform([pred_class])[0], prob

# Пример
for msg in test_messages:
    label, prob = predict_loaded(msg)
    print(f"Сообщение: {msg}")
    print(f"Класс (загруженная модель): {label}, вероятность спама: {prob:.4f}\n")
```

---



## 6. Код с загруженной моделью для быстрого запуска

```python
# Укажите ваши ссылки
url_model = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/spam_model.h5"
url_tokenizer = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/tokenizer.pickle"
url_encoder = "https://github.com/ваш_логин/spam-detector-lstm/raw/main/label_encoder.pickle"

!wget -O spam_model.h5 {url_model}
!wget -O tokenizer.pickle {url_tokenizer}
!wget -O label_encoder.pickle {url_encoder}

import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

loaded_model = load_model('spam_model.h5')
with open('tokenizer.pickle', 'rb') as f: tokenizer = pickle.load(f)
with open('label_encoder.pickle', 'rb') as f: label_encoder = pickle.load(f)

MAX_LENGTH = 100  # должно совпадать

def predict_spam_fast(text):
    cleaned = str(text).lower()
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    prob = loaded_model.predict(padded, verbose=0)[0][0]
    return label_encoder.inverse_transform([int(prob > 0.5)])[0], prob

# Пример
msg = "Вы выиграли iPhone! Заберите приз по ссылке..."
print(predict_spam_fast(msg))
```

---

## 8. Задания для студентов

### Задание 1. Подготовка репозитория и данных
- Создайте публичный репозиторий на GitHub.
- Выберите один из предложенных датасетов (рекомендуется SMS Spam Collection) и загрузите его в репозиторий под именем `spam_data.csv`.
- Получите raw‑ссылку на файл.

### Задание 2. Подготовка данных в ноутбуке
- Загрузите данные из репозитория по ссылке.
- Выведите статистику: количество примеров, распределение по классам, гистограмму длин сообщений.
- Выполните очистку текста (приведение к нижнему регистру, удаление лишних символов).
- Выполните токенизацию и паддинг последовательностей.
- Разделите данные на обучающую и тестовую выборки (80/20) со стратификацией.

### Задание 3. Построение модели
- Постройте модель с одним слоем LSTM (можно добавить Dropout).
- Выведите summary модели.
- Скомпилируйте модель с оптимизатором 'adam' и функцией потерь 'binary_crossentropy'.

### Задание 4. Обучение модели с фиксацией времени (обязательно)
- Обучите модель с использованием EarlyStopping (patience=5) на 30 эпохах.
- **Обязательно** добавьте код для измерения времени обучения (например, `start_time = time.time()`, после обучения `end_time - start_time`) и выведите результат в консоль.
- Постройте графики потерь и точности на обучающей и валидационной выборках.

### Задание 5. Оценка качества
- Предскажите классы на тестовой выборке.
- Выведите classification report (precision, recall, F1) и матрицу ошибок.
- Проанализируйте, какие типы ошибок допускает модель.

### Задание 6. Функция предсказания
- Реализуйте функцию `predict_spam`, которая принимает текст и возвращает метку (spam/ham) и вероятность спама.
- Протестируйте на 3-5 собственных примерах (включая явный спам и обычное сообщение).

### Задание 7. Сохранение и загрузка в репозиторий
- Сохраните модель, токенизатор и label encoder.
- Загрузите эти файлы в свой репозиторий.
- Получите raw‑ссылки на каждый файл.

### Задание 8. Загрузка модели из репозитория и быстрый инференс
- В отдельной ячейке загрузите файлы из репозитория.
- Восстановите модель и выполните предсказание для тех же примеров, что и в задании 6.
- Убедитесь, что результаты совпадают.

### Задание 9. Анализ результатов
- Оцените, насколько хорошо модель отличает спам от обычных сообщений.
- Предложите способы улучшения модели (например, использование двунаправленного LSTM, увеличение размера словаря, добавление дополнительных признаков).

### Задание 10*. Дополнительно (по желанию)
- Реализуйте модель с двунаправленным LSTM (Bidirectional).
- Попробуйте использовать предобученные эмбеддинги (Word2Vec, FastText).
- Добавьте в предобработку удаление стоп-слов и лемматизацию. Сравните результаты.

---

## 9. Заключение

В ходе выполнения этого задания вы:
- создали собственный репозиторий на GitHub и научились загружать туда файлы;
- познакомились с основными этапами создания модели выявления спама на основе LSTM;
- подготовили размеченные данные, выполнили токенизацию и паддинг;
- построили и обучили рекуррентную нейронную сеть для бинарной классификации;
- **обязательно зафиксировали время обучения модели**;
- оценили качество модели с помощью метрик классификации;
- освоили сохранение модели и вспомогательных объектов, а затем их загрузку из репозитория для повторного использования.

Полученные навыки являются основой для решения широкого круга задач классификации текстов, включая фильтрацию спама, анализ тональности, определение тематики и другие.

---